# GraphUnslopify: local figures with evidence

Install `graphunslopify[notebook]` in this kernel's Python environment. This notebook creates public synthetic data and uses the packaged local translator. After installation it needs no hosted translator or download. Run all cells in order in JupyterLab or a compatible notebook editor.

We choose the axes, units, experimental repeat IDs and claim ourselves. Suggestions cannot establish experimental independence.

In [ ]:
import sys
from pathlib import Path
from tempfile import mkdtemp

import pandas as pd
from IPython.display import Image, display

from graphunslopify import Figure, SpecError

print("Kernel:", sys.executable)

WORK = Path(mkdtemp(prefix="graphunslopify-notebook-"))
rows = [
    {"epoch": epoch, "score": 50 + epoch + seed + gain, "model": model, "seed": seed}
    for epoch in range(1, 7)
    for model, gain in [("base", 0), ("ours", 2)]
    for seed in range(4)
]
data = pd.DataFrame(rows)
data_path = WORK / "results.csv"
data.to_csv(data_path, index=False)
assert len(data) == 48
data.head()

## Describe intent and render locally

One observation unit is a named seed within an epoch/model. The generator owns colors, line styles, typography and exports. The builder also has named constructors for all 21 supported kinds; kind-specific options can be set with `.set(...)`.

In [ ]:
figure = (
    Figure.line(data_path)
    .x("epoch", "Epoch")
    .y("score", "Score")
    .by("model", order=["base", "ours"])
    .average_over("seed")
    .claim("beats_everywhere", "ours", "base")
    .set(legend={"position": "outside_bottom"})
    .named("learning-curve")
    .output(formats=["png", "svg", "pdf"], dpi=150)
)
result = figure.render(cwd=WORK / "figures", strict=True, api="local")
report = result["report"]
assert report["status"] == "passed"
assert report["data"]["counts"]["sample_sizes"] == [4] * 12
assert report["claims"][0]["verdict"] == "holds"
display(Image(filename=str(WORK / "figures" / "learning-curve.png")))

## Read the evidence

`holds` is a literal comparison of the plotted summaries. It does not establish statistical significance or equivalence. This example intentionally has overlapping intervals; the report and description disclose that caveat. Strict mode rejects failed or untested required checks, while ordinary warnings have a separate policy.

In [ ]:
assert report["claims"][0]["interval_overlap"][0] > 0
assert "intervals overlap" in report["alt_text"]
print(report["alt_text"])
report["data"]["counts"]

## Keep a reproducible specification and script

These files point to `results.csv` beside them. Copy the whole example directory if you want to rerun it elsewhere. `graphunslopify run learning-curve.py --strict --json` runs the standalone program; `graphunslopify audit-init` and `audit` can extend this into a paper manifest.

In [ ]:
portable = Figure(figure.to_dict())
portable.spec["data"]["path"] = "results.csv"
(WORK / "curve.figure.json").write_text(portable.to_json(), encoding="utf8")
translated = portable.translate(api="local")
assert translated["ok"]
(WORK / "learning-curve.py").write_text(translated["code"], encoding="utf8")
print(f"Example files: {WORK}")

## A false claim must be rejected

Reversing the comparison changes the claim, not the data. Catch `SpecError` and inspect its report so a notebook cannot silently turn a failed assertion into an approved figure.

In [ ]:
counterexample = Figure(figure.to_dict()).named("counterexample")
counterexample.spec["claims"][0].update(subject="base", reference="ours")
try:
    counterexample.render(cwd=WORK / "rejected", strict=True, api="local")
except SpecError as problem:
    assert problem.report is not None
    assert problem.report["status"] == "failed"
    assert problem.report["claims"][0]["verdict"] == "fails"
    print("Expected rejection: the reversed claim fails.")
else:
    raise AssertionError("A false claim unexpectedly passed")